# Tutorial 3: Operators, Composition and Sensitivity Kernels

The first two tutorials treated the sea level equation as a solver: hand it a load, get
back a set of fields. This tutorial takes the other view that `pyslfp` supports, in which
the linearised sea level equation is a **linear operator** between Hilbert spaces.

Writing

$$ A : \zeta \longmapsto (\xi, u, \phi, \omega) $$

for the map taking a direct surface load $\zeta$ to the relative sea level change $\xi$,
the vertical displacement, the gravitational potential change and the angular velocity
change, makes two things available that the solver alone does not.

The first is **composition**. Most quantities of interest are not the raw fields but
functionals of them: a global mean, a regional average, a value at a tide gauge, a
spherical harmonic coefficient of the potential. Each of these is itself a linear
operator, so the quantity of interest is a product of operators that can be assembled
piece by piece and then treated as a single object.

The second is the **adjoint**. If $F$ is such a composition, mapping loads to a vector of
$n$ numbers, then its adjoint $F^{*}$ maps $\mathbb{R}^{n}$ back into the load space, and

$$ \langle F\zeta, y\rangle_{\mathbb{R}^n} = \langle \zeta, F^{*}y\rangle_{L}
\qquad\text{for all } \zeta, y. $$

Taking $y$ to be the $i$th standard basis vector shows that $F^{*}e_i$ is the function
$K_i$ for which

$$ (F\zeta)_i = \int_{\partial M} K_i \, \zeta \, \mathrm{d}S. $$

That function is the **sensitivity kernel** of the $i$th datum: it says how much that
datum changes per unit of mass added at each point on the Earth's surface. One adjoint
solve produces the whole kernel, whatever the dimension of the load space. The theory
behind the adjoint of the sea level equation, and the reciprocity relations that make it
computable, is set out in
[Al-Attar et al. (2024)](https://academic.oup.com/gji/article/236/1/362/7338265).

The operator machinery comes from
[`pygeoinf`](https://github.com/da380/pygeoinf), which supplies the Hilbert spaces,
the `LinearOperator` class, and the composition and adjoint algebra used below.

In [ ]:
# import the libraries either locally or installing when on colab
import numpy as np
import matplotlib.pyplot as plt

try:
    import pyslfp as sl
except ImportError:
    %pip install pyslfp --quiet
    import pyslfp as sl

from cartopy import crs as ccrs

from pyslfp.linear_operators import (
    FingerPrintOperator,
    averaging_operator,
    ocean_average_operator,
    remove_ocean_average_operator,
    ice_sheet_basis_operator,
    ice_thickness_change_to_load_operator,
    lebesgue_load_space,
    TideGaugeObservationModel,
)

## 1. The fingerprint operator

`FingerPrintOperator` wraps `LinearSeaLevelEquation` as a `pygeoinf` `LinearOperator`. It
is built in the same way as the solver, and `from_defaults` again gives a PREM Earth model
with present-day ICE-7G as the background state.

A truncation degree of 128 is used throughout. Everything here works at 256 as well and
gives sharper maps; it is simply slower, and the patterns of interest are all long
wavelength.

In [ ]:
LMAX = 128

fingerprint = FingerPrintOperator.from_defaults(lmax=LMAX)

# The background state and the non-dimensionalisation are reached through the operator.
state = fingerprint.state
params = state.model.parameters

The operator carries its domain and codomain with it. The domain is the space of
square-integrable surface loads on the sphere. The codomain is a direct sum of four
spaces, one for each component of the response: three fields and a two-dimensional
Euclidean space for the angular velocity change.

In [ ]:
response_space = fingerprint.codomain
field_space = response_space.subspace(0)

print("Domain:  ", type(fingerprint.domain).__name__, "of dimension", fingerprint.domain.dim)
print("Codomain:", type(response_space).__name__, "of dimension", response_space.dim)
print()
for i, subspace in enumerate(response_space.subspaces):
    print(f"  component {i}: {type(subspace).__name__:15s} dimension {subspace.dim}")

Applying the operator solves the sea level equation. The result is a list holding the four
components, in the same order as the solver returns them, so the calculation of Tutorial 1
becomes a single function call. The colour scale below is clipped at one metre; the sea
level fall next to West Antarctica reaches several times that, and leaving the scale
unclipped hides everything else.

In [ ]:
load = state.west_antarctic_load(fraction=0.1)
sea_level_change, displacement, potential_change, angular_velocity_change = fingerprint(load)

fig, ax = sl.create_map_figure(figsize=(10, 5))
sl.plot(
    sea_level_change * state.ocean_projection() * params.length_scale,
    ax=ax,
    vmin=-1.0,
    vmax=1.0,
    colorbar_kwargs={"label": "Sea level change (m)"},
)
plt.show()

### A note on spaces

By default the load space is a Lebesgue space, so its inner product is the plain $L^2$
integral over the sphere and the adjoints computed below are $L^2$ adjoints. Passing
`load_parameters` and `response_parameters` instead gives Sobolev spaces, whose inner
products penalise roughness. That choice changes the adjoint, and Section 4 returns to
what this means in practice. Elliptic regularity means the response is one Sobolev order
smoother than the load, so the response order may not exceed the load order plus one.

Units remain non-dimensional. Loads are converted with `params.load_scale`, lengths with
`params.length_scale`, and masses with `params.mass_scale`.

## 2. Composing operators

The response space is a direct sum, and `subspace_projection(i)` is the operator that
extracts the $i$th component. Composing it with the fingerprint gives an operator that
maps a load straight to the sea level change.

Composition uses the `@` operator throughout, and each product is itself a
`LinearOperator` with a domain, a codomain and an adjoint.

In [ ]:
sea_level = response_space.subspace_projection(0)

# Load -> sea level change.
sea_level_response = sea_level @ fingerprint

print(sea_level_response.domain.dim, "->", sea_level_response.codomain.dim)

### Global mean sea level

`ocean_average_operator` averages a field over the oceans. Placing it at the end of the
chain gives an operator mapping a load to a single number, the barystatic contribution to
global mean sea level.

In [ ]:
mean_sea_level = ocean_average_operator(state, field_space) @ sea_level @ fingerprint

datum = mean_sea_level(load)
print(f"Global mean sea level change: {datum[0] * params.length_scale * 1000:.1f} mm")

### Departure from the global mean

The interesting part of a fingerprint is not the mean but the pattern about it.
`remove_ocean_average_operator` subtracts the ocean average from a field, and inserting it
before the projection isolates that pattern.

In [ ]:
anomaly = remove_ocean_average_operator(state, field_space) @ sea_level @ fingerprint

anomaly_field = anomaly(load)

# By construction the result has no ocean average left in it.
residual = ocean_average_operator(state, field_space)(anomaly_field)[0]
print(f"Ocean average of the result: {residual * params.length_scale * 1000:.2e} mm")

# The near field runs well off this scale, which is chosen to show the far field.
fig, ax = sl.create_map_figure(figsize=(10, 5))
sl.plot(
    anomaly_field * state.ocean_projection() * params.length_scale,
    ax=ax,
    vmin=-0.5,
    vmax=0.5,
    colorbar_kwargs={"label": "Sea level change relative to global mean (m)"},
)
plt.show()

### A single site

`averaging_operator` takes a list of weighting functions and returns the operator giving
the average of a field over each of them, normalised by the area of the weight. A disk
load of unit amplitude serves as the weight, so the following measures sea level averaged
over a cap of radius four degrees, relative to the global mean. This stands in for what a
tide gauge records once the global signal has been removed.

In [ ]:
SITE = (40.7, -74.0)  # New York

cap = state.disk_load(4.0, SITE[0], SITE[1], 1.0)
local_average = averaging_operator(state, field_space, [cap])

local_anomaly = (
    local_average
    @ remove_ocean_average_operator(state, field_space)
    @ sea_level
    @ fingerprint
)

value = local_anomaly(load)[0] * params.length_scale * 1000
print(f"Local sea level relative to the global mean: {value:.1f} mm")

### Changing the input variable

Operators can be attached at the front of the chain as well. Ice sheet models produce
thickness changes rather than loads, and `ice_thickness_change_to_load_operator` performs
the conversion, applying the ice density and masking out the oceans.

`ice_sheet_basis_operator` goes one step further. It maps a small vector of coefficients
to the field that is uniform on each of a set of basin groupings, so composing the three
gives an operator from a handful of numbers to global mean sea level. Applying it to the
standard basis vectors gives the sensitivity of global mean sea level to a uniform metre
of thickening over each region. Since the input and output are both lengths, the ratio is
dimensionless and needs only a factor of 1000 to be read in millimetres per metre.

In [ ]:
ice_space = lebesgue_load_space(state.model)

basins = ice_sheet_basis_operator(state, ice_space, groupings="macro_regions")
ice_to_load = ice_thickness_change_to_load_operator(state, ice_space, fingerprint.domain)

basin_to_mean_sea_level = mean_sea_level @ ice_to_load @ basins

labels = ["West Antarctica", "East Antarctica", "Antarctic Peninsula", "Greenland"]
identity = np.eye(basin_to_mean_sea_level.domain.dim)

for i, label in enumerate(labels):
    response = basin_to_mean_sea_level(identity[i])[0] * 1000
    print(f"{label:22s} {response:8.2f} mm of global mean sea level per metre of thickening")

## 3. Adjoints

Every operator above carries an adjoint, formed automatically as the chain is built. For
the fingerprint operator itself the adjoint is not a transpose of a stored matrix — no
matrix is ever formed — but a second sea level calculation, in which the components of the
response act as forcings on the generalised equation. This is the reciprocity result of
Al-Attar et al. (2024), and it is why `SeaLevelEquation` provides
`solve_generalised_equation` alongside the ordinary solver.

The defining property is
$\langle A\zeta, r\rangle = \langle \zeta, A^{*}r\rangle$. `pygeoinf` will check this,
together with linearity, on random draws from measures that the operator supplies for the
purpose.

In [ ]:
fingerprint.check(
    n_checks=2,
    check_rtol=1e-4,
    domain_measure=fingerprint.load_measure_for_testing(),
    codomain_measure=fingerprint.response_measure_for_testing(),
)

### Kernels from adjoints

For an operator $F$ mapping loads to $n$ numbers, $F^{*}e_i$ is the sensitivity kernel of
the $i$th datum. Start with the global mean, where the answer is known in advance: mass
conservation with a fixed shoreline requires

$$ \rho_w A_o \, \bar{\xi} + \int_{\partial M} \zeta \, \mathrm{d}S = 0, $$

so the kernel should be the constant $-1/(\rho_w A_o)$, independent of where the mass is
placed.

In [ ]:
kernel = mean_sea_level.adjoint(np.array([1.0]))

print(f"Kernel minimum: {kernel.data.min():.12e}")
print(f"Kernel maximum: {kernel.data.max():.12e}")
print(f"Predicted:      {-1.0 / (params.water_density * state.ocean_area):.12e}")

The kernel is constant to machine precision, which is the statement that global mean sea
level responds only to the total mass added and not at all to its distribution.

Pairing the kernel with a load must reproduce the forward calculation.

In [ ]:
print("From the kernel: ", fingerprint.domain.inner_product(kernel, load))
print("From the solver: ", mean_sea_level(load)[0])

The two agree to machine precision here. That is worth a word of caution for chains built
differently. Operators such as `ice_thickness_change_to_load_operator` and the projections
work by multiplying by a mask, and the product of two band-limited fields is not itself
band-limited. Where the masks have sharp edges, the truncation back to degree `lmax`
aliases, and the adjoint identity then holds to the accuracy of the discretisation rather
than to machine precision. The discrepancy falls as `lmax` rises, but it is a property of
the representation, not a defect in the adjoint.

### A kernel with structure

The local functional built in Section 2 is a more useful case. Its kernel says how much
sea level at the site changes, relative to the global mean, per unit of mass added
anywhere on the Earth's surface.

Multiplying the kernel by a mass gives the resulting datum, so the natural way to
dimensionalise it is to choose a mass and read the map in the units of the datum. Taking
1000 gigatonnes — roughly three years of present-day ice loss, and about 2.8 mm of global
mean sea level — puts the map in millimetres.

In [ ]:
gigatonne = 1.0e12 / params.mass_scale


# Scales a non-dimensional sea level kernel to mm of sea level per 1000 Gt of mass.
def kernel_in_mm_per_1000_gigatonnes(k):
    return k * (1000 * gigatonne) * params.length_scale * 1000


local_kernel = kernel_in_mm_per_1000_gigatonnes(local_anomaly.adjoint(np.array([1.0])))

print(f"Range: {local_kernel.data.min():.2f} to {local_kernel.data.max():.2f} mm per 1000 Gt")

# The near field dominates, so the colour scale is clipped to show the far field.
fig, ax = sl.create_map_figure(figsize=(10, 5))
sl.plot(
    local_kernel,
    ax=ax,
    vmin=-5.0,
    vmax=5.0,
    colorbar_kwargs={"label": "mm of local sea level per 1000 Gt of added mass"},
)
plt.show()

Two features stand out. Mass added close to the site raises sea level there strongly,
through direct gravitational attraction, and this near-field peak runs off the clipped
colour scale. Mass added anywhere far away lowers sea level at the site relative to the
global mean, because the site takes less than its share of the water released elsewhere.
The transition sits at a few thousand kilometres, which is the reason that a tide gauge is
a much better instrument for the ice that is near it than for the ice that is not.

The map is one adjoint solve. Obtaining the same information forwards would mean placing a
unit mass at every point of the grid in turn and solving each time.

### Polar wander

Nothing restricts the argument to sea level. The fourth component of the response is the
angular velocity change, and its kernel follows in the same way. Scaling by $b/\Omega$
converts an angular velocity perturbation into the displacement of the rotation pole at
the Earth's surface.

In [ ]:
pole = response_space.subspace_projection(3) @ fingerprint

pole_factor = params.mean_sea_floor_radius / params.rotation_frequency

shift = pole(load) * pole_factor * params.length_scale
print(f"Pole displacement for this melt scenario: {np.linalg.norm(shift):.1f} m")

fig, axes = plt.subplots(
    1, 2, figsize=(13, 4), subplot_kw={"projection": ccrs.Robinson()}, layout="constrained"
)

for ax, i, label in zip(axes, [0, 1], ["x axis", "y axis"]):
    unit = np.zeros(2)
    unit[i] = 1.0
    k = kernel_in_mm_per_1000_gigatonnes(pole.adjoint(unit) * pole_factor)
    sl.plot(
        k,
        ax=ax,
        symmetric=True,
        colorbar_kwargs={"label": f"Pole shift towards {label} (mm per 1000 Gt)"},
    )

plt.show()

Both kernels are spherical harmonics of degree two and order one, which is exactly what
the theory of rotational feedbacks predicts: only that part of the load can exert a torque
that reorients the rotation axis. The nodal lines show where mass may be added without
moving the pole at all, and the extrema show where it is most effective.

Note also that these kernels are pure long wavelength patterns, whereas the sea level
kernel above was sharply peaked. This difference is the reason that polar motion and tide
gauges carry genuinely complementary information about the same load.

## 4. Sobolev spaces and point observations

The kernels so far have all been $L^2$ adjoints of averaging functionals, which are
well behaved. Point evaluation is not. Evaluating a field at a single point is an
unbounded functional on $L^2$ — no square-integrable kernel represents it — so a load
space with a stronger inner product is needed. On the sphere it suffices to work in a
Sobolev space of order greater than one, and `check_load_space` enforces exactly this
whenever an operator asks for point values.

Passing `load_parameters` and `response_parameters` as `(order, relative scale)` builds
the operator over Sobolev spaces, the scale being measured in Earth radii.

In [ ]:
sobolev_fingerprint = FingerPrintOperator.from_defaults(
    lmax=LMAX,
    load_parameters=(2.0, 0.05),
    response_parameters=(2.0, 0.05),
)

print("Load space:", type(sobolev_fingerprint.domain).__name__,
      "of order", sobolev_fingerprint.domain.order)

`TideGaugeObservationModel` assembles the point evaluation and the fingerprint into a
single forward operator. Its `from_gloss_network` constructor uses the GLOSS station
network, downloading the coordinates on first use.

In [ ]:
gauges = TideGaugeObservationModel.from_gloss_network(sobolev_fingerprint)
print(f"{len(gauges.points)} stations")

data = gauges.forward_operator(load) * params.length_scale * 1000

fig, ax = sl.create_map_figure(figsize=(10, 5))
sl.plot_points(
    gauges.points,
    data=data,
    ax=ax,
    s=25,
    symmetric=True,
    colorbar=True,
    colorbar_kwargs={"label": "Sea level change (mm)"},
)
plt.show()

The adjoint of this operator gives, for each station, the function representing that
station's reading. It is worth being careful about what this function is. In an $L^2$ load
space the adjoint returns the sensitivity kernel, the function that reproduces the datum
under the $L^2$ pairing. Here the load space is a Sobolev space, so the adjoint returns
the representer with respect to the **Sobolev** inner product, which is a smoothed
version of the kernel with a width set by the scale supplied when the space was built.

This is not a numerical compromise but the correct object for the space in question. In a
regularised inversion the load is sought in the Sobolev space, and the Sobolev representer
is what determines the resolution actually achieved. Changing the order or the scale
changes it, and comparing the two makes the effect of a prior visible before any data are
inverted.

In [ ]:
# The station nearest the site used earlier.
distances = [(lat - SITE[0]) ** 2 + (lon - SITE[1]) ** 2 for lat, lon in gauges.points]
index = int(np.argmin(distances))
print("Station:", gauges.names[index], gauges.points[index])

unit = np.zeros(len(gauges.points))
unit[index] = 1.0
representer = gauges.forward_operator.adjoint(unit)

fig, ax = sl.create_map_figure(figsize=(10, 5))
sl.plot(
    representer,
    ax=ax,
    symmetric=0.3,
    colorbar_kwargs={"label": "Sobolev representer (non-dimensional)"},
)
plt.show()

## 5. Where this leads

Three things have been used repeatedly. The fingerprint is a linear operator with a domain
and a codomain; operators compose with `@` to build whatever functional is of interest;
and the adjoint of the composition is formed automatically and yields sensitivity kernels
at the cost of one solve per datum.

The same pattern underlies the rest of `pyslfp.linear_operators`. `GraceObservationModel`
replaces point evaluation with `to_coefficient_operator`, mapping the potential change to
spherical harmonic coefficients, and `WMBMethod` provides the purely spectral
approximation of Wahr, Molenaar and Bryan (1998) for comparison and preconditioning.
`AltimetryObservationModel` and `JointAltimetryObservationModel` handle sea surface height
over the oceans and over the ice sheets. `joint_ice_ocean_to_load_operator` widens the
domain when ice and ocean contributions are to be inferred together.

Because these forward operators are `pygeoinf` objects, they can be handed directly to
that library's inference machinery. Sensitivity kernels are the natural diagnostic to look
at first: they show what a network can and cannot see, and they do so without any data.